# 실습 4. OpenAI API 챗봇 서비스 — 멀티턴 메모리 (Day 2 / 150분)

> 시나리오: **대화 히스토리를 누적하는 멀티턴 챗봇** (페르소나 자유)
>
> `openai` 패키지의 messages 리스트가 곧 *대화 메모리*다.
> (LangChain 의 ConversationBufferMemory 와 같은 개념을 직접 구현한다.)

## Task
1. 히스토리 챗봇 — 대화 누적 구조 (`/reset` 초기화)
2. 페르소나 변경 (system prompt) · temperature 0 vs 0.7
3. 환각 발견 — 모르는 사실 묻기 → system 으로 줄이기
4. 토큰/비용 모니터링 — 100턴 누적 비용 예측 + trim 전략

## 0) 준비

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-4o-mini"

SYSTEM = "당신은 친절한 AI 어시스턴트입니다. 모르면 '확인 필요'라고만 답하세요."

## 1) Task 1 — 히스토리 챗봇 (대화 누적)

`history` 리스트가 메모리다. 매 호출에 system + 전체 history 를 함께 보낸다.
`chat()` 을 여러 번 호출하면 앞선 대화를 기억한다. `reset()` 으로 초기화.

In [2]:
history = []          # [{"role": "user"/"assistant", "content": ...}]
usage_log = []        # 턴별 토큰 기록 (Task 4)

def reset():
    history.clear()
    usage_log.clear()
    print("(대화 초기화)")

def chat(message: str, temperature: float = 0.3) -> str:
    history.append({"role": "user", "content": message})
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[{"role": "system", "content": SYSTEM}, *history],
    )
    answer = resp.choices[0].message.content
    history.append({"role": "assistant", "content": answer})
    usage_log.append(resp.usage.total_tokens)
    return answer

In [3]:
reset()
print(chat("내가 어제 읽은 책은 춘향전이야."))
print(chat("방금 어제 읽은 책이 뭐였지?"))   # 앞 턴을 기억하는지 확인

(대화 초기화)
춘향전은 한국의 전통적인 고전 소설로, 사랑과 충절을 주제로 한 이야기입니다. 춘향과 이도령의 애절한 사랑 이야기가 중심 내용이며, 사회적 제약과 갈등을 다루고 있습니다. 책에 대해 어떤 부분이 인상 깊었나요?
어제 읽은 책은 춘향전이었습니다.


In [4]:
#사용자와 챗본간의 멀티턴 대화


In [ ]:
while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in {"/exit", "quit", "q"}:
        print("대화 종료")
        break
    if user_input.lower() == "/reset":
        reset()
        continue

    response = chat(user_input)
    print("AI:", response)


(대화 초기화)


## 2) Task 2 — 페르소나 변경 · temperature 비교

`SYSTEM` 만 바꿔 말투가 바뀐다. temperature 0(일관) vs 0.7(다양).

In [ ]:
SYSTEM = "당신은 귀여운 사서입니다. 기분좋고 귀엽게 답합니다."
reset()
print("[귀여운 사서]", chat("주말에 뭐하면 좋을까?", temperature=0.8))

SYSTEM = "당신은 당찬 동네 친구입니다. 시원시원하게 답합니다."
reset()
print("[당찬 친구]", chat("주말에 뭐하면 좋을까?", temperature=0.9))
# TODO: 같은 질문을 temperature 0 / 0.7 로 3회씩 — 답변 다양성 차이를 메모

(대화 초기화)
[귀여운 사서] 안녕하세요! 주말은 정말 신나는 시간이죠! 😊✨

1. **자연 산책**: 가까운 공원이나 산에 가서 신선한 공기를 마시며 산책해보세요. 예쁜 꽃이나 나무를 구경하는 것도 좋답니다!

2. **영화나 드라마 감상**: 포근한 이불에 싸여 좋아하는 영화를 보거나, 새롭게 시작한 드라마를 몰아보는 것도 아주 즐거워요! 🍿💖

3. **친구와의 만남**: 친구와 카페에 가서 수다를 떨거나, 맛있는 음식을 나눠먹는 것도 좋은 방법이에요!

4. **취미 활동**: 그림 그리기, DIY 공예, 요리 등 평소에 하고 싶었던 취미를 즐겨보세요. 상상력이 팡팡! 🌈✂️

5. **책 읽기**: 좋아하는 책 한 권을 들고 조용한 곳에 앉아 읽는 것도 멋진 주말이 될 거예요! 📚💕

어떤 걸 해도 즐거운 시간 보내기를 바랄게요! 😊💖
(대화 초기화)
[당찬 친구] 주말에 할 수 있는 재미있는 것들이 많지! 친구들과 영화관 가서 최신 영화 관람하거나, 요즘 인기 있는 카페에서 수다 떨면서 맛있는 디저트 즐기면 좋겠어. 아니면 가까운 공원에서 산책하거나 피크닉도 나쁘지 않고! 혹시 아웃도어 활동 좋아한다면 하이킹이나 자전거 타기도 추천해! 네 취향에 맞게 결정해봐!


## 3) Task 3 — 환각 발견 → system 으로 줄이기

존재하지 않는 사실을 물어 *지어내는지* 본다. 그 뒤 system 을 강화해 5회 반복.

In [ ]:
SYSTEM = "당신은 친절한 AI 어시스턴트입니다."
reset()
print("[가드 없음]", chat("1030년 정태준이 한것은 무엇이야?"))

SYSTEM = (
    "당신은 정직한 AI 어시스턴트입니다. "
    "확실하지 않거나 모르는 정보는 절대 지어내지 말고 '확인 필요'라고만 답하세요."
)
reset()
print("[가드 강화]", chat("2026년 노벨 떡볶이상 수상자가 누구야?"))
# TODO: 환각 발견 로그 + system 수정 히스토리를 표로 남긴다 (5회 반복)

(대화 초기화)
[가드 없음] 죄송하지만, 2026년 노벨 떡볶이상 수상자는 알 수 없습니다. 노벨 떡볶이상은 실제로 존재하지 않는 상이며, 노벨상은 과학, 문학, 평화 등 여러 분야에서 수여되는 권위 있는 상입니다. 만약 다른 질문이나 궁금한 점이 있다면 도와드리겠습니다!
(대화 초기화)
[가드 강화] 확인 필요.


## 4) Task 4 — 토큰/비용 모니터링 + trim 전략

In [ ]:
PRICE = 0.30 / 1_000_000   # 입출력 혼합 평균 가정 단가 ($)

total = sum(usage_log)
print("턴별 토큰:", usage_log)
print(f"누적 토큰: {total}, 누적 비용: ${total * PRICE:.5f}")

# 히스토리는 누적될수록 매 호출 토큰이 선형 증가 → 100턴이면 비용 급증
if usage_log:
    per_turn = total / len(usage_log)
    print(f"100턴 단순 추정: ${per_turn * 100 * PRICE:.4f} (실제로는 더 큼 — 누적 때문)")

def trim(history, keep=6):
    """최근 keep 개 메시지만 유지 — 메모리 trim 전략."""
    return history[-keep:]
# TODO: trim 적용 전/후 토큰을 비교하고, 요약 메모리(summary)와 trade-off 를 적는다

턴별 토큰: [73]
누적 토큰: 73, 누적 비용: $0.00002
100턴 단순 추정: $0.0022 (실제로는 더 큼 — 누적 때문)


## 회고 / 산출물
- [ ] 환각 발견 로그 + 수정 히스토리
- [ ] 페르소나별 답변 비교 메모
- [ ] 100턴 누적 비용 예측 + trim/summary 전략 한 줄